# S1 · 01 · De los chatbots a los agentes de IA
El bucle de un agente y tu primer `Agent` con salida tipada. Las secciones 1 a 7 son ejemplos mínimos de las **Demos 2 y 3**; tu trabajo son los **Ejercicios 1 y 2** (≈ 14 min).

Corre sin red ni LLM con el modelo simulado (`COURSE_OFFLINE=1`, por defecto).


## Preparación
Las dependencias se instalan **antes** de abrir el notebook (`uv sync`). Si algo falla, `uv run python -m utilidades.doctor` dice qué parte no responde.
Por defecto el notebook usa el **modelo simulado** (`utilidades/simulado.py`): no es IA y **no mide calidad**; sirve para ver el recorrido sin red ni cuota. Para IA real: configura tu ruta con `uv run python -m utilidades.configurar`, pon `COURSE_OFFLINE=0` en `.env` y reinicia el kernel.


In [ ]:
from utilidades.practica import preparar, revisar, ver_mensajes

settings, OFFLINE = preparar()


## 1 · ¿Con qué modelo trabajo hoy?
El mismo agente corre con cuatro «motores»; solo cambia una línea de `.env` (repasa `s0/02_repaso.ipynb` si «token» o «temperatura» no te suenan):

| Ruta | `.env` | Datos |
|---|---|---|
| Gemini gratis | `MODE=gemini` + `GEMINI_API_KEY` (ver [docs/gemini.md](../docs/gemini.md)) | Viajan a Google; solo datos ficticios |
| OpenRouter gratis | `MODE=openrouter` + `OPENROUTER_API_KEY` + un ID `:free` (ver [docs/openrouter.md](../docs/openrouter.md)) | Viajan al proveedor del modelo |
| Ollama local | `MODE=local` + `OLLAMA_MODEL=gemma4:e2b` (ver [docs/ollama.md](../docs/ollama.md)) | No salen de tu equipo |
| Simulado | `COURSE_OFFLINE=1` (por defecto) | Sin IA real |

`crear_modelo` (en `utilidades/config.py`) solo devuelve **una clase de Pydantic AI** según la ruta (`OpenAIChatModel` con `OllamaProvider`, `GoogleModel` u `OpenRouterModel`): eso es lo que recibe `Agent(...)`. Si la ruta dice «simulado» y quieres IA real, pon `COURSE_OFFLINE=0` en `.env` y reinicia el kernel.


In [ ]:
from utilidades.config import crear_modelo, describir_ruta

modelo = crear_modelo(settings, OFFLINE)
print("Modelo activo:", describir_ruta(settings, OFFLINE), "→", type(modelo).__name__)
if not OFFLINE and settings.mode == "gemini":
    print("Gemini (capa gratuita): si aparece 429, espera un minuto o cambia GEMINI_MODEL.")


## 2 · Un agente desde cero: ReAct en unas 40 líneas
**ReAct** (Yao et al., 2023) intercala *razonar*, *actuar* y *observar*. Antes de usar un framework escribimos el bucle en Python puro, como en la práctica de DeepLearning.AI. Necesitamos cuatro piezas: una tool, un «modelo», un despachador y el bucle.
Primero la **tool** (una función normal de Python) y un **«modelo» de juguete**: una función que devuelve JSON. Pide la tool la primera vez y, cuando ve el resultado, responde. No entiende nada: solo nos deja ver el bucle sin red.


In [ ]:
import json
import re



def calcular_presupuesto(horas: float, tarifa: float) -> dict:
    """Calcula el costo en Bs según horas y tarifa por hora."""
    return {"total": horas * tarifa, "moneda": "Bs"}


PEDIDO = "Calcula el presupuesto de 3 horas de taller a 40 Bs por hora."
TOOLS = {"calcular_presupuesto": calcular_presupuesto}


def modelo_juguete(mensajes: list[dict]) -> str:
    """Un «modelo» de mentira: devuelve JSON. No es IA."""
    ultimo = mensajes[-1]
    if ultimo["role"] == "tool":
        datos = json.loads(ultimo["content"])
        return json.dumps({"respuesta": f"El presupuesto es {datos['total']} {datos['moneda']}."})
    horas, tarifa = re.findall(r"\d+(?:\.\d+)?", ultimo["content"])[:2]
    return json.dumps(
        {
            "pensamiento": "Debo multiplicar horas por tarifa",
            "tool": "calcular_presupuesto",
            "args": {"horas": float(horas), "tarifa": float(tarifa)},
        }
    )


print(modelo_juguete([{"role": "user", "content": PEDIDO}]))


Ahora el **despachador** (ejecuta la tool que pidió el modelo y devuelve su resultado como texto) y el **bucle**: pedir al modelo, ejecutar, devolver el resultado y repetir hasta que responda. `max_pasos` es el control más barato contra un bucle infinito.


In [ ]:
def ejecutar_tool(pedido: dict) -> str:
    funcion = TOOLS[pedido["tool"]]
    return json.dumps(funcion(**pedido["args"]), ensure_ascii=False)


def agente(pedido: str, modelo=modelo_juguete, despachar=ejecutar_tool, max_pasos: int = 5) -> str:
    mensajes = [{"role": "user", "content": pedido}]
    for _ in range(max_pasos):
        salida = json.loads(modelo(mensajes))
        if "respuesta" in salida:
            return salida["respuesta"]
        print(f"💭 {salida.get('pensamiento', '')}")
        print(f"🤖 pide tool {salida['tool']}({salida['args']})")
        observacion = despachar(salida)
        print(f"🔧 resultado {observacion}")
        mensajes.append({"role": "assistant", "content": json.dumps(salida)})
        mensajes.append({"role": "tool", "content": observacion})
    raise RuntimeError(f"Sin respuesta tras {max_pasos} pasos")


print("✅", agente(PEDIDO))


Eso es un agente: un **modelo que pide**, un **programa que valida y ejecuta**, y un **ciclo con límite**. Todo lo demás (esquemas, reintentos, historial, aprobación humana) son mejoras a este mismo esqueleto.


### 2.1 · Con un LLM real: `/chat/completions`
Ollama, Gemini y OpenRouter ofrecen una API compatible con OpenAI. El protocolo de tools es el mismo para casi todos los proveedores: enviamos `tools` (un JSON Schema por función), el modelo responde con `tool_calls` y nosotros contestamos con mensajes de rol `tool`. Fíjate en que **el esquema se escribe a mano**: es lo primero que un framework nos ahorra.
Esta celda solo corre con IA real (`COURSE_OFFLINE=0`) y usa la ruta de tu `.env` (`ruta_openai` devuelve URL, clave y modelo según `MODE`). Si el modelo no responde, imprime una pista y el notebook sigue. Una versión con una sola interfaz de modelo está en `s1/bucle_sin_framework.py`.


In [ ]:
if not OFFLINE:
    import httpx

    from utilidades.config import ruta_openai

    HERRAMIENTAS = [
        {
            "type": "function",
            "function": {
                "name": "calcular_presupuesto",
                "description": calcular_presupuesto.__doc__,
                "parameters": {
                    "type": "object",
                    "properties": {"horas": {"type": "number"}, "tarifa": {"type": "number"}},
                    "required": ["horas", "tarifa"],
                },
            },
        }
    ]

    def agente_llm(pedido: str, max_pasos: int = 5) -> str:
        base, clave, nombre = ruta_openai(settings)
        mensajes = [{"role": "user", "content": pedido}]
        for _ in range(max_pasos):
            cuerpo = {
                "model": nombre,
                "messages": mensajes,
                "tools": HERRAMIENTAS,
                "temperature": 0,
            }
            r = httpx.post(
                f"{base}/chat/completions",
                headers={"Authorization": f"Bearer {clave}"},
                json=cuerpo,
                timeout=120,
            )
            r.raise_for_status()
            mensaje = r.json()["choices"][0]["message"]
            llamadas = mensaje.get("tool_calls") or []
            if not llamadas:
                return mensaje["content"]
            mensajes.append(
                {"role": "assistant", "content": mensaje.get("content") or "", "tool_calls": llamadas}
            )
            for llamada in llamadas:
                nombre, args = llamada["function"]["name"], llamada["function"]["arguments"]
                args = json.loads(args) if isinstance(args, str) else args
                print(f"🤖 pide tool {nombre}({args})")
                observacion = ejecutar_tool({"tool": nombre, "args": args})
                print(f"🔧 resultado {observacion}")
                mensajes.append(
                    {"role": "tool", "tool_call_id": llamada["id"], "content": observacion}
                )
        raise RuntimeError(f"Sin respuesta tras {max_pasos} pasos")

    try:
        print("✅", agente_llm(PEDIDO))
    except (httpx.HTTPError, ValueError) as exc:
        print(f"El modelo no respondió ({type(exc).__name__}: {str(exc)[:120]}). Prueba: uv run python -m utilidades.doctor --live")


### 2.2 · Lo mismo con Pydantic AI
Con el framework, el esquema sale de la **firma y el docstring** de la función, los argumentos se validan, el bucle y sus límites ya existen y los mensajes quedan tipados. `ver_mensajes` muestra el mismo recorrido que imprimimos a mano.


In [ ]:
from pydantic_ai import Agent
from pydantic_ai.tools import Tool

agente_pai = Agent(
    modelo,
    instructions="Responde en español y breve. Usa las herramientas para calcular.",
    tools=[calcular_presupuesto],
)
resultado = await agente_pai.run(PEDIDO)
ver_mensajes(resultado)
print(resultado.output)


Compara con el esquema que escribimos a mano en 2.1: Pydantic AI lo **derivó** de `calcular_presupuesto(horas: float, tarifa: float)`.


In [ ]:
print(Tool(calcular_presupuesto).tool_def.parameters_json_schema)


## 3 · El primer agente de Mi copiloto
Un agente son tres piezas: un **modelo** (`modelo`), unas **instrucciones** y un **contrato de salida** (`Respuesta`, un modelo de Pydantic). Todavía no tiene tools ni memoria: las tools llegan en `02_herramientas.ipynb`.
Lee las instrucciones como un contrato de comportamiento: dicen qué hacer, qué no inventar y cuándo pedir un dato.


In [ ]:
from pydantic import BaseModel, Field


class Respuesta(BaseModel):
    texto: str = Field(min_length=1, description="Respuesta en español, breve")
    fuentes: list[str] = Field(default_factory=list, description="Fuentes que respaldan la respuesta")
    requiere_revision: bool = Field(default=False, description="True si hay una acción por aprobar")


INSTRUCCIONES = """Eres Mi copiloto, el asistente personal de Alex. Responde en español, breve.
Si falta la información, responde: «No tengo ese dato». No inventes datos.
Si falta un dato para planificar, dilo."""

print(Respuesta.model_json_schema()["properties"].keys())


`output_type=Respuesta` hace que el agente devuelva un objeto validado. Con Ollama (sin tools) conviene `NativeOutput(Respuesta)`, que fuerza JSON; los demás proveedores usan la clase directa. `modo_salida` decide por ti y lo vas a reutilizar en los ejercicios.


In [ ]:
from pydantic_ai import NativeOutput


def modo_salida(tipo):
    """Ollama sin tools admite JSON forzado (NativeOutput); los demás, la clase directa."""
    return NativeOutput(tipo) if getattr(modelo, "system", "") == "ollama" else tipo


Ejecutamos una petición con objetivo, restricciones y un criterio de éxito. `resultado.output` es un objeto `Respuesta` ya validado; `resultado.usage` cuenta requests y tokens; `ver_mensajes` muestra el recorrido.


In [ ]:
basico = Agent(modelo, instructions=INSTRUCCIONES, output_type=modo_salida(Respuesta))
PETICION = (
    "Soy Alex. Hoy tengo una cita a las 09:00 y quiero leer 60 minutos y caminar 30. "
    "Propón tres bloques sin solapar la cita e indica qué dato falta para planificar bien."
)
r_basico = await basico.run(PETICION)
print(r_basico.output.model_dump())
print(r_basico.usage)
ver_mensajes(r_basico)


Los **mensajes** son la memoria de corto plazo de una ejecución: `all_messages()` devuelve la lista completa (petición, respuesta del modelo, resultados de tools). Inspecciona los tipos; no imprimas historiales con datos reales.


In [ ]:
print([type(m).__name__ for m in r_basico.all_messages()])


## 4 · Respuesta libre y respuesta tipada
Sin `output_type` el agente devuelve **texto**; con `output_type=Respuesta` devuelve un objeto validado. Lo tipado cuesta algo más (el esquema viaja al modelo y la respuesta lleva campos), pero tu interfaz recibe una estructura estable en lugar de un párrafo que hay que interpretar.


In [ ]:
libre = Agent(modelo, instructions=INSTRUCCIONES)
r_libre = await libre.run(PETICION)
print("libre  :", type(r_libre.output).__name__, "·", r_libre.usage)
print("tipado :", type(r_basico.output).__name__, "·", r_basico.usage)


## 5 · Modos de salida: Tool, Native y Prompted
Pydantic AI puede pedir la salida estructurada de tres maneras, y eso se decide con la **clase** que pasas a `output_type`:

| Clase | Cómo funciona | Se usa en |
|---|---|---|
| `Respuesta` (por defecto) | El esquema es una **tool de salida** | Casi todos los proveedores |
| `NativeOutput(Respuesta)` | **JSON nativo** forzado por el proveedor | Ollama **sin** tools |
| `PromptedOutput(Respuesta)` | El **esquema va en el prompt** y Pydantic AI valida | Ollama **con** tools (notebook 02) |

Con Ollama, forzar JSON por gramática **y** pedir tools a la vez falla; por eso con tools se usa `PromptedOutput`.


In [ ]:
print("este agente usa:", type(modo_salida(Respuesta)).__name__)


## 6 · Un esquema válido no garantiza un dato verdadero
`Respuesta` exige que `texto` no esté vacío, que `fuentes` sea una lista y que `requiere_revision` sea booleano. Nada más. Un modelo puede cumplir el contrato y **mentir**: la validación revisa la forma; la verdad se comprueba con tools, fuentes y evaluaciones (el notebook 02 y la sesión 6).


In [ ]:
inventada = Respuesta(
    texto="Tu reunión con Ana es mañana a las 15:00 en la sala azul.",
    fuentes=["agenda:inventada"],
)
print("¿válida?", Respuesta.model_validate(inventada.model_dump()) == inventada)
print("¿verdadera? Este agente no tiene ninguna tool que lo confirme.")


## 7 · Leer los errores: ¿falló el modelo o falló el programa?
- **Falla el programa** (entorno, red, clave): `ConnectError`, `UserError`, `ModuleNotFoundError`. Se arregla fuera del modelo.
- **Falla el modelo** (salida inválida): Pydantic AI devuelve el error al modelo para que reintente; si agota los reintentos lanza `UnexpectedModelBehavior`.
- **Salta un límite** que tú pusiste: `UsageLimitExceeded`. Es el control funcionando, no un fallo.
Aquí provocamos los dos últimos sin red, con un modelo que siempre responde mal.


In [ ]:
from pydantic_ai import UnexpectedModelBehavior, UsageLimitExceeded, UsageLimits
from pydantic_ai.messages import ModelResponse, ToolCallPart
from pydantic_ai.models.function import FunctionModel


def modelo_que_falla(mensajes, info):
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, {"texto": ""})])


agente_fallido = Agent(FunctionModel(modelo_que_falla), output_type=Respuesta, retries=1)
try:
    await agente_fallido.run("Hola")
except UnexpectedModelBehavior as exc:
    print("El modelo falló →", type(exc).__name__, "·", exc)
try:
    await basico.run("Hola", usage_limits=UsageLimits(request_limit=0))
except UsageLimitExceeded as exc:
    print("Un límite saltó →", type(exc).__name__)


## Ejercicio 1 · Tu propio `output_type`: `PlanDia`
**Se apoya en:** Demo 3 · **Nivel:** básico · ≈ 6 min. **Qué cambia:** en la demo el contrato era `Respuesta`; aquí diseñas otro, con una lista anidada y un campo opcional.

Define un contrato para un plan del día: una **lista de bloques** (hora y actividad) y un `dato_faltante` opcional (`str | None`) para cuando falta información. Crea un agente con ese `output_type` y comprueba con `isinstance` que la salida es un `PlanDia`. Mira `Respuesta` arriba: es un pariente de lo que vas a escribir.


In [ ]:
from pydantic import BaseModel, Field


class Bloque(BaseModel):
    hora: str = Field(description="HH:MM")
    actividad: str


class PlanDia(BaseModel):
    # TODO Ejercicio 1: añade los campos `bloques` (lista de Bloque) y `dato_faltante` (str | None, por defecto None).
    pass


agente_plan = Agent(modelo, instructions=INSTRUCCIONES, output_type=modo_salida(PlanDia))
r1 = await agente_plan.run(PETICION)
print(r1.output)


In [ ]:
revisar(
    "Ejercicio 1 · PlanDia tiene bloques y dato_faltante",
    lambda: {"bloques", "dato_faltante"} <= set(PlanDia.model_fields),
    "Añade los campos bloques y dato_faltante",
)
revisar(
    "Ejercicio 1 · acepta un plan válido",
    lambda: PlanDia(
        bloques=[Bloque(hora="09:00", actividad="Cita")], dato_faltante="duración de la cita"
    ).bloques[0].hora
    == "09:00",
    "bloques es una lista de Bloque y dato_faltante es opcional",
)
revisar("Ejercicio 1 · el agente devuelve un PlanDia", lambda: isinstance(r1.output, PlanDia), "")


## Ejercicio 2 · Un despachador que sobrevive a los errores
**Se apoya en:** Demo 2 · **Nivel:** medio · ≈ 8 min. **Qué cambia:** en la demo el modelo siempre pedía bien; aquí un modelo «travieso» pide una tool que no existe y tu programa no debe caerse.

Un modelo real a veces **pide una tool que no existe** o con argumentos mal formados. Con `ejecutar_tool`, el programa se cae con un `KeyError`. Escribe `despachar_seguro`: debe devolver siempre un texto JSON; si la tool no existe o los argumentos fallan, devuelve `{"error": "..."}` como observación para que el modelo pueda corregirse. Es la idea de `ModelRetry` en Pydantic AI.


In [ ]:
def modelo_travieso(mensajes: list[dict]) -> str:
    """Primero pide una tool inventada; si ve un error, corrige y pide la correcta."""
    ultimo = mensajes[-1]
    if ultimo["role"] == "user":
        return json.dumps({"pensamiento": "Pruebo una tool", "tool": "presupuestar", "args": {}})
    if "error" in ultimo["content"]:
        return modelo_juguete(mensajes[:1])
    return modelo_juguete(mensajes)


def despachar_seguro(pedido: dict) -> str:
    # TODO Ejercicio 2: captura la tool inexistente y los argumentos inválidos y devuelve {"error": "..."} en JSON.
    return ejecutar_tool(pedido)


try:
    respuesta_2 = agente(PEDIDO, modelo_travieso, despachar=despachar_seguro)
except Exception as exc:  # mientras el ejercicio esté incompleto
    respuesta_2 = f"{type(exc).__name__}: {exc}"
print(respuesta_2)


In [ ]:
revisar(
    "Ejercicio 2 · la tool inexistente devuelve un error en JSON",
    lambda: "error" in json.loads(despachar_seguro({"tool": "presupuestar", "args": {}})),
    "Captura KeyError y devuelve json.dumps({'error': ...})",
)
revisar(
    "Ejercicio 2 · argumentos inválidos también",
    lambda: "error" in json.loads(despachar_seguro({"tool": "calcular_presupuesto", "args": {"x": 1}})),
    "Captura TypeError (argumentos que la función no acepta)",
)
revisar(
    "Ejercicio 2 · el agente se recupera y responde 120",
    lambda: "120" in respuesta_2,
    "Con el despachador seguro el modelo ve el error, corrige y responde",
)


## Evidencia y cierre
Guarda en tu entrega: tu `PlanDia`, el despachador seguro y una frase sobre lo que **no** pudo comprobar la validación del esquema. Entrega el notebook con tus cambios y, si puedes, con las salidas de la ejecución con IA real.
**¿Qué puede hacer ahora Mi copiloto que antes no podía?** Responde en español con instrucciones propias y devuelve objetos validados (`Respuesta`, `PlanDia`). Todavía **no consulta nada**: no ve la agenda ni el correo de Alex. Si le preguntas «¿qué tengo mañana?», solo puede reconocer que le falta el dato.
**Siguiente:** `02_herramientas.ipynb`: le damos herramientas (agenda, correo, clima y crear eventos) y vemos cómo decide, cómo se corrige y cómo pide permiso antes de escribir.
**Preguntas para pensar:** ¿qué parte de tu `agente()` de Python puro hace hoy Pydantic AI por ti? ¿Qué control añadirías antes de dejar que el agente ejecute una acción que escribe?
